In [4]:
import os
import pyspark

print("Répertoire :", os.getcwd())
print("Version PySpark :", pyspark.__version__)

Répertoire : /home/jovyan/work
Version PySpark : 3.5.3


# TP : Mise en oeuvre d’un pipeline ETL et d’un Data Lakehouse (Bronze/Silver/Gold)

## Preparation de l'envirronnement

### 1.1 Initialisation de la session Spark

In [5]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Eshop-ETL-Lakehouse")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Spark master :", spark.sparkContext.master)
print("Application ID :", spark.sparkContext.applicationId)


Spark version : 3.5.3
Spark master : local[*]
Application ID : local-1794080282534


### 1.2 Ingestion des données sources

In [6]:
customers_raw =(
    spark.read.format("csv")
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "false")
    .load("data/source/customers.csv")
)

customers_raw.printSchema()
customers_raw.show(10, truncate=False)

print("Nombre de clients :", customers_raw.count())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
|C006       |Amine 06     |Nord  |2024-02-12       |
|C007       |Alice 07     |Sud   |2024-02-19       |
|C008       |Youssef 08   |Centre|2024-02-26       |
|C009       |Sara 09      |Est   |2024-03-04       |
|C010       |Omar 10      |Ouest |2024-03-11       |
+-----------+-------------+------+-----------------+
only showing top 10 rows

Nombre de clients : 32


In [7]:
# Let's continue the ingestion process by reading the other data sources
def read_csv_data(filename):
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("delimiter", ";")
        .option("inferSchema", "false")
        .csv(f"data/source/{filename}")
    )

products_raw = read_csv_data("products.csv")
orders_raw = read_csv_data("orders.csv")
order_items_raw = read_csv_data("order_items.csv")


In [8]:
#promotions file
promotions_raw = spark.read.text("data/source/promotions.jsonl")
promotions_raw.printSchema()
promotions_raw.show(truncate=False)

root
 |-- value: string (nullable = true)

+----------------------------------------------------------------------------------------------------+
|value                                                                                               |
+----------------------------------------------------------------------------------------------------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025-01-01","end_date":"2025-12-31"}  |
|{"promotion_code":"PROMO10","discount_rate":0.1,"start_date":"2025-05-01","end_date":"2025-05-31"}  |
|{"promotion_code":"SUMMER15","discount_rate":0.15,"start_date":"2025-06-01","end_date":"2025-08-31"}|
|{"promotion_code":"VIP20","discount_rate":0.2,"start_date":"2025-01-01","end_date":"2025-12-31"}    |
|{"promotion_code":"OLD30","discount_rate":0.3,"start_date":"2024-01-01","end_date":"2024-12-31"}    |
|{"promotion_code":"BAD_JSON","discount_rate":0.10,"start_date":"2025-06-01","end_date":             |
+-----------------------------

In [9]:
# number of lines in each source
sources = {
    "customers": customers_raw,
    "products": products_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
    "promotions": promotions_raw
}

for name, df in sources.items():
    print(f"{name}: {df.count()} lignes")


customers: 32 lignes
products: 14 lignes
orders: 44 lignes
order_items: 86 lignes
promotions: 6 lignes


### Data Exploration

In [10]:
for name, df in sources.items():
    print(f"\n===== {name.upper()} =====")

    df.printSchema()
    df.show(5, truncate=False)


===== CUSTOMERS =====
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
+-----------+-------------+------+-----------------+
only showing top 5 rows


===== PRODUCTS =====
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- unit_cost: string (nullable = true)

+----------+------------+------------+----------+--------

In [11]:
# Volumes verification

expected_counts = {
    "customers": 32,
    "products": 14,
    "orders": 44,
    "order_items": 86,
    "promotions": 6
}

for name, df in sources.items():
    actual = df.count()
    expected = expected_counts[name]

    print(
        f"{name}: attendu={expected}, "
        f"obtenu={actual}"
    )

    assert actual == expected, (
        f"Volume incorrect pour {name}"
    )

customers: attendu=32, obtenu=32
products: attendu=14, obtenu=14
orders: attendu=44, obtenu=44
order_items: attendu=86, obtenu=86
promotions: attendu=6, obtenu=6



### Conclusion de l'ingestion

Les cinq sources ont été chargées dans des DataFrames Spark.

Les champs CSV ont été conservés sous forme de chaînes
de caractères afin de préserver leurs valeurs originales.

Le fichier JSONL a été lu en mode texte pour conserver
également les lignes mal formées.

Les volumes ont été comparés aux valeurs de référence.

Les contrôles de qualité, les conversions de types et la
gestion des anomalies seront réalisés dans la couche Silver.


# Bronze Zone

In [12]:
# Original files consservation 

from pathlib import Path
import shutil

project_dir = Path("/home/jovyan/work")

source_dir = project_dir / "data" / "source"
bronze_dir = project_dir / "data_lakehouse" / "bronze" / "original"

bronze_dir.mkdir(parents=True, exist_ok=True)

filenames = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
    "promotions.jsonl"
]

for filename in filenames:
    source_file = source_dir / filename
    destination_file = bronze_dir / filename

    shutil.copy2(source_file, destination_file)

    print(f"Copié : {filename}")


Copié : customers.csv
Copié : products.csv
Copié : orders.csv
Copié : order_items.csv
Copié : promotions.jsonl


In [13]:
# Copy intgrity verification

import hashlib

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename in filenames:
    original = source_dir / filename
    bronze = bronze_dir / filename

    assert file_hash(original) == file_hash(bronze)

    print(f"{filename}: copie identique")

customers.csv: copie identique
products.csv: copie identique
orders.csv: copie identique
order_items.csv: copie identique
promotions.jsonl: copie identique


#### Json manifest

In [14]:
from uuid import uuid4
from datetime import datetime, timezone

run_id = str(uuid4())

print("Run ID :", run_id)
print("Date :", datetime.now(timezone.utc).isoformat())

Run ID : 4c37d8f5-4a59-4a10-a8a7-e964c6a7538f
Date : 2026-11-08T00:04:25.149571+00:00


In [15]:
import json
import shutil

manifest = []

for filename in filenames:
    source_path = source_dir / filename
    bronze_path = bronze_dir / filename

    record = {
        "run_id": run_id,
        "source": filename,
        "ingestion_time": datetime.now(timezone.utc).isoformat(),
        "bronze_path": str(bronze_path),
        "row_count": None,
        "status": "failed"
    }

    try:
        shutil.copy2(source_path, bronze_path)

        assert file_hash(source_path) == file_hash(bronze_path)

        if filename.endswith(".jsonl"):
            df = spark.read.text(str(bronze_path))
        else:
            df = (
                spark.read
                .option("header", True)
                .option("sep", ";")
                .option("inferSchema", False)
                .csv(str(bronze_path))
            )

        record["row_count"] = df.count()
        record["sha256"] = file_hash(bronze_path)
        record["status"] = "success"

    except Exception as e:
        record["error"] = str(e)

    manifest.append(record)

    print(filename, record["status"], record["row_count"])


customers.csv success 32
products.csv success 14
orders.csv success 44
order_items.csv success 86
promotions.jsonl success 6


In [16]:
metadata_dir = project_dir / "data_lakehouse" / "metadata" / "ingestion"

metadata_dir.mkdir(parents=True, exist_ok=True)

manifest_path = metadata_dir / f"manifest_{run_id}.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Manifeste enregistré :", manifest_path)

assert all(record["status"] == "success" for record in manifest)

Manifeste enregistré : /home/jovyan/work/data_lakehouse/metadata/ingestion/manifest_4c37d8f5-4a59-4a10-a8a7-e964c6a7538f.json


##### Manifest verification

In [17]:
from pathlib import Path
import json

metadata_dir = Path(
    "/home/jovyan/work/data_lakehouse/metadata/ingestion"
)

manifests = sorted(
    metadata_dir.glob("manifest_*.json"),
    key=lambda p: p.stat().st_mtime
)

if manifests:
    latest_manifest = manifests[-1]

    with open(latest_manifest, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        print(
            record["source"],
            record["row_count"],
            record["status"],
            record["run_id"]
        )
else:
    print("Aucun manifeste trouvé.")

customers.csv 32 success 4c37d8f5-4a59-4a10-a8a7-e964c6a7538f
products.csv 14 success 4c37d8f5-4a59-4a10-a8a7-e964c6a7538f
orders.csv 44 success 4c37d8f5-4a59-4a10-a8a7-e964c6a7538f
order_items.csv 86 success 4c37d8f5-4a59-4a10-a8a7-e964c6a7538f
promotions.jsonl 6 success 4c37d8f5-4a59-4a10-a8a7-e964c6a7538f


###  Verification de la relecture Bronze

In [20]:
# test in customers file
customers_bronze = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(str(bronze_dir / "customers.csv"))
)

customers_count = customers_bronze.count()

print("Nombre de lignes Bronze :", customers_count)

assert customers_count == 32

Nombre de lignes Bronze : 32


In [24]:
# other files 
for record in manifest:

    filename = record["source"]
    path = record["bronze_path"]
    expected_count = record["row_count"]

    if filename.endswith(".jsonl"):
        df = spark.read.text(path)
    else:
        df = (
            spark.read
            .option("header", True)
            .option("sep", ";")
            .option("inferSchema", False)
            .csv(path)
        )

    actual_count = df.count()

    assert actual_count == expected_count, (
        f"Volume incorrect pour {filename}"
    )

    print(f"{filename} : {actual_count} lignes  OK")

customers.csv : 32 lignes  OK
products.csv : 14 lignes  OK
orders.csv : 44 lignes  OK
order_items.csv : 86 lignes  OK
promotions.jsonl : 6 lignes  OK
